# qGAN Experiment Tutorial

Use this notebook to run or load one selected experiment, run or load a battery, and analyze completed checkpoints.


## 1. Setup

Run this notebook from inside the repository or from `qgan/notebooks`. The cell below makes `qgan/src` importable without installing the package.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
qgan_dir = None

for candidate in (cwd, *cwd.parents):
    if (candidate / "src" / "qgan_v2").is_dir() and (candidate / "configs").is_dir():
        qgan_dir = candidate
        break
    if (candidate / "qgan" / "src" / "qgan_v2").is_dir():
        qgan_dir = candidate / "qgan"
        break

if qgan_dir is None:
    raise RuntimeError("Could not find qgan project directory from current working directory.")

repo_root = qgan_dir.parent
src_path = qgan_dir / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("qgan_dir:", qgan_dir)
print("repo_root:", repo_root)


qgan_dir: /home/benat/Qiskit-IBM/qgan
repo_root: /home/benat/Qiskit-IBM


In [2]:
from qgan_v2.config.battery import create_battery_configs
from qgan_v2.config.loader import load_config_file, load_run_config
from qgan_v2.implementations.registry import IMPLEMENTATIONS
from qgan_v2.main import run_battery, run_train
from qgan_v2.storage.paths import get_run_path, get_training_data_filename, resolve_data_path
from qgan_v2.training.data import load_training_data_file
from qgan_v2.visualization import get_visual_config, run_visualization

state = None
battery_states = {}
battery_results = []
valid_config_files = []

sorted(IMPLEMENTATIONS)


['manual_estimator', 'qml_torch', 'runtime_packed']

## 2. IBM Runtime Credentials

This cell is optional. Enable it only when you need to save or verify IBM Runtime credentials.


In [3]:
save_account = False
check_runtime_account = False

if save_account or check_runtime_account:
    from getpass import getpass
    from qiskit_ibm_runtime import QiskitRuntimeService

if save_account:
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token=getpass("IBM Quantum API token: "),
        instance=None,
        set_as_default=True,
        overwrite=True,
        verify=True,
    )

if check_runtime_account:
    service = QiskitRuntimeService()
    print("Available backends:", len(service.backends()))


## 3. Single Experiment

The first YAML file in `configs/singles` is selected by default. Set `single_config_path` to choose another one.


In [4]:
single_config_path = repo_root / "qgan/data/train/base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0/config.yaml"

single_config_files = sorted((qgan_dir / "configs" / "singles").glob("*.yaml"))
if single_config_path is None:
    if not single_config_files:
        raise FileNotFoundError("No single config files found in qgan/configs/singles.")
    single_config_path = single_config_files[0]

config_path = Path(single_config_path)
config = load_run_config(config_path)
training_data_file = get_training_data_filename(config)
state = None

print("single config:", config_path)
print("run id:", config["run"]["id"])
print("checkpoint:", training_data_file)
print("checkpoint_exists:", training_data_file.exists())


single config: /home/benat/Qiskit-IBM/qgan/data/train/base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0/config.yaml
run id: base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0
checkpoint: /home/benat/Qiskit-IBM/qgan/data/train/base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0/training_data.pth
checkpoint_exists: True


### Run Or Load

Choose one boolean. Function arguments stay in the function call.


In [5]:
run_single_experiment = False

if run_single_experiment:
    state = run_train(str(config_path), reset_data=False)
else:
    if not training_data_file.exists():
        raise FileNotFoundError(f"No checkpoint found: {training_data_file}")
    state = load_training_data_file(training_data_file)
    print("loaded state:", training_data_file)

loaded state: /home/benat/Qiskit-IBM/qgan/data/train/base-qml_torch-q4-real-SPSA-aerCPU-rand0-seed0/training_data.pth


### Inspect Checkpoint


In [6]:
print("current_epoch:", state.current_epoch)
print("best_eval:", state.metrics.best_eval())

current_epoch: 238
best_eval: 3.032940695344694


### Visualize


In [7]:
visualize_experiment_results = False

if training_data_file.exists() and visualize_experiment_results:
    run_visualization(
        config_path,
        get_visual_config({
            "draw_circuits": False,
            "draw_hardware_layout": True,
            "draw_probs": False,
            "draw_images": True,
            "draw_results": True,
        }),
    )


## 4. Battery Experiments

The first YAML file in `configs/batteries` is selected by default. Set `battery_path` to choose another one.


In [8]:
battery_path = repo_root / "qgan/configs/batteries/train/train_times_gpu.yaml"

battery_files = sorted((qgan_dir / "configs" / "batteries").glob("**/*.yaml"))
if battery_path is None:
    if not battery_files:
        raise FileNotFoundError("No battery config files found in qgan/configs/batteries.")
    battery_path = battery_files[0]

battery_path = Path(battery_path)
print("battery:", battery_path)


battery: /home/benat/Qiskit-IBM/qgan/configs/batteries/train/train_times_gpu.yaml


### Run Or Load

Set `run_battery_experiments = True` to run the battery. Leave it `False` to load existing checkpoints. In both cases, `battery_states` is loaded from checkpoint files.


In [9]:
run_battery_experiments = False

battery_states = {}
battery_results = []
valid_config_files = []
invalid_config_files = []
missing_checkpoints = []

if run_battery_experiments:
    battery_results = run_battery(
        battery_path,
        reset_data=False,
        reset_rb=False,
        stop_on_error=False,
        overwrite=False,
        keep_states=False,
    )
    config_files = [Path(result["config_file"]) for result in battery_results]
else:
    battery_data_path = resolve_data_path(load_config_file(battery_path)["default_config_values"]["run"]["data_path"])
    config_files = sorted(battery_data_path.glob("*/config.yaml"))

for path in config_files:
    try:
        cfg = load_run_config(path)
    except Exception as exc:
        invalid_config_files.append((path, exc))
        continue

    valid_config_files.append(path)
    checkpoint = get_training_data_filename(cfg)
    if checkpoint.exists():
        battery_states[cfg["run"]["id"]] = load_training_data_file(checkpoint)
    else:
        missing_checkpoints.append(checkpoint)

print("config files:", len(config_files))
print("valid config files:", len(valid_config_files))
print("invalid config files:", len(invalid_config_files))
print("loaded states:", len(battery_states))
print("missing checkpoints:", len(missing_checkpoints))


config files: 265
valid config files: 265
invalid config files: 0
loaded states: 237
missing checkpoints: 28


### Battery Summary


In [10]:
for path in valid_config_files[:20]:
    cfg = load_run_config(path)
    checkpoint = get_training_data_filename(cfg)
    run_id = cfg["run"]["id"]
    print(
        run_id,
        "done=" + str(checkpoint.exists()),
        "loaded=" + str(run_id in battery_states),
    )

remaining = len(valid_config_files) - 20
if remaining > 0:
    print("...", remaining, "more configs")


amp-qml_torch-q16-noiseless-PSR-aerCPU-rand0-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-PSR-aerCPU-rand1-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-PSR-aerGPU-rand0-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-PSR-aerGPU-rand1-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-REG-aerCPU-rand0-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-REG-aerCPU-rand1-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-REG-aerGPU-rand0-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-REG-aerGPU-rand1-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-SPSA-aerCPU-rand0-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-SPSA-aerCPU-rand1-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-SPSA-aerGPU-rand0-seed0 done=False loaded=False
amp-qml_torch-q16-noiseless-SPSA-aerGPU-rand1-seed0 done=False loaded=False
amp-qml_torch-q16-noisy-PSR-aerCPU-rand0-seed0 done=False loaded=False
amp-qml_torch-q16-noisy-P

## 5. Implementation Adapters

Quick view of implementations in the battery configs loaded above.


In [11]:
for path in valid_config_files[:20]:
    cfg = load_run_config(path)
    print(path.relative_to(qgan_dir))
    print("  implementation:", cfg["implementation"]["name"])
    print("  preset:", cfg["experiment"]["implementation"])
    print("  gradient:", cfg["experiment"]["gradient_method"])
    print()

remaining = len(valid_config_files) - 20
if remaining > 0:
    print("...", remaining, "more configs")


data/train/times/amp-qml_torch-q16-noiseless-PSR-aerCPU-rand0-seed0/config.yaml
  implementation: qml_torch
  preset: amp
  gradient: PSR

data/train/times/amp-qml_torch-q16-noiseless-PSR-aerCPU-rand1-seed0/config.yaml
  implementation: qml_torch
  preset: amp
  gradient: PSR

data/train/times/amp-qml_torch-q16-noiseless-PSR-aerGPU-rand0-seed0/config.yaml
  implementation: qml_torch
  preset: amp
  gradient: PSR

data/train/times/amp-qml_torch-q16-noiseless-PSR-aerGPU-rand1-seed0/config.yaml
  implementation: qml_torch
  preset: amp
  gradient: PSR

data/train/times/amp-qml_torch-q16-noiseless-REG-aerCPU-rand0-seed0/config.yaml
  implementation: qml_torch
  preset: amp
  gradient: REG

data/train/times/amp-qml_torch-q16-noiseless-REG-aerCPU-rand1-seed0/config.yaml
  implementation: qml_torch
  preset: amp
  gradient: REG

data/train/times/amp-qml_torch-q16-noiseless-REG-aerGPU-rand0-seed0/config.yaml
  implementation: qml_torch
  preset: amp
  gradient: REG

data/train/times/amp-qml_to

## 6. Thesis Results Analysis

The results follow this order: time and feasibility; presets; execution type; gradient method; randomness; scaling; and implementation/packing validation.

Unless stated otherwise, quality figures use completed 1000-epoch simulator runs. CPU/GPU copies of the same scientific run are collapsed for quality analysis, while device remains explicit in timing. Learning-dynamics figures use the original qGAN arrangement: generator and discriminator losses share the upper panel and evaluation occupies the lower panel. Generator loss, discriminator loss, and evaluation retain the original blue, red, and orange color families. Multiple compared groups use related shades; faint run traces and median/IQR shadows show variation across seeds.


### Analysis setup and figure export

Set `EXPORT_FIGURES = True` to save PDF and 300-dpi PNG versions under `qgan/figures/thesis_results`. KL divergence (`base`) and image-gradient scores (`ang`/`amp`) are faceted rather than pooled because their scales are not comparable.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qgan_v2.analysis import (
    SCIENTIFIC_PAIR_FIELDS,
    evaluation_metric_groups,
    factor_sweep_groups,
    filter_results,
    grouped_performance_table,
    load_results,
    plot_feasibility_matrix,
    plot_final_metric,
    plot_metric_by_category_field,
    plot_metric_by_numeric_field,
    plot_paired_delta,
    plot_training_dynamics_comparison,
    results_table,
    save_figure,
    select_completed_results,
    select_main_convergence_results,
    select_usable_results,
    truncate_results,
    unique_values,
)
from qgan_v2.visualization import plot_generated_output

EXPORT_FIGURES = False
FIGURE_DIR = qgan_dir / "figures" / "thesis_results"
GPU_MODEL_LABEL = "RTX6000"  # Change if another GPU timing battery is loaded.


def finish_figure(fig, stem):
    fig.tight_layout()
    if EXPORT_FIGURES:
        for saved_path in save_figure(fig, FIGURE_DIR, stem):
            print("saved:", saved_path)
    plt.show()
    plt.close(fig)


def show_rows(rows, columns=None, limit=100):
    rows = list(rows)
    visible = rows[:limit]
    if columns is not None:
        visible = [{column: row.get(column) for column in columns} for row in visible]
    try:
        import pandas as pd
        from IPython.display import display

        display(pd.DataFrame(visible))
    except ImportError:
        for row in visible:
            print(row)
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more rows")


def dynamics_figure(runs, compare_by, title, stem):
    fig, axes = plot_training_dynamics_comparison(
        runs,
        compare_by=compare_by,
        center="median",
        spread="iqr",
    )
    fig.suptitle(title)
    finish_figure(fig, stem)
    return fig, axes


def best_results_figure(runs, compare_by, title, stem):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    plot_final_metric(
        runs, compare_by=compare_by, metric_name="best_eval",
        point_color_by="seed", ax=axes[0],
    )
    axes[0].set_title("Best evaluation")
    plot_final_metric(
        runs, compare_by=compare_by, metric_name="epoch_of_best_eval",
        point_color_by="seed", ax=axes[1],
    )
    axes[1].set_title("Epoch of best evaluation")
    fig.suptitle(title)
    finish_figure(fig, stem)
    return fig, axes


raw_convergence = load_results(qgan_dir / "data" / "train")
raw_timing = load_results(qgan_dir / "data" / "train" / "times")
main_convergence = select_main_convergence_results(raw_convergence, expected_epochs=1000)
timing_results = select_completed_results(raw_timing, expected_epochs=5)
hardware_results = select_usable_results(
    raw_convergence, execution_types=("real",)
)
validation_results = select_usable_results(
    raw_convergence, execution_types=("fake_real",)
)

print("raw convergence/case-study configs:", len(raw_convergence))
print("completed, deduplicated simulator runs:", len(main_convergence))
print("completed five-epoch timing runs:", len(timing_results))
print("usable real-hardware runs:", len(hardware_results))
print("usable implementation-validation runs:", len(validation_results))
print("evaluation families:", {
    name: len(runs) for name, runs in evaluation_metric_groups(main_convergence).items()
})


### 1. Time, Cost, and Feasibility

Timing comes first so the reader knows which experiments were computationally practical before interpreting model quality.


#### 1.1 Time analysis

These panels use completed five-epoch timing batteries. Median time per epoch is reported because it is less sensitive to a slow initialization epoch. The factors are preset, execution type, gradient method, and `rand0`/`rand1`.

The timing environments are the one-CPU battery (`CPU/1`), the explicit four-CPU battery (`CPU/4`), and the configured GPU battery. Every series is plotted on the exact categorical tick—there is no horizontal series offset.


In [ ]:
primary_timing = [
    run for run in timing_results
    if (
        run.metadata.get("simulator_device") == "GPU"
        or run.metadata.get("run_device") == "CPU"
    )
    and run.metadata.get("label") in (None, "cpu4")
]
for run in primary_timing:
    if run.metadata.get("simulator_device") == "GPU":
        environment = f"GPU ({GPU_MODEL_LABEL})"
    elif run.metadata.get("label") == "cpu4":
        environment = "CPU/4"
    else:
        environment = "CPU/1"
    run.metadata["timing_environment"] = environment

print("timing environments:", unique_values(primary_timing, "timing_environment"))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
plot_metric_by_category_field(
    primary_timing,
    x_field="preset",
    metric_name="median_time_per_epoch",
    line_by="timing_environment",
    filters={
        "implementation": "qml_torch", "execution_type": "noiseless",
        "gradient_method": "SPSA", "n_qubits": 4, "randomness": 0,
    },
    ax=axes[0, 0],
)
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("Timing by preset")

plot_metric_by_category_field(
    primary_timing,
    x_field="execution_type",
    metric_name="median_time_per_epoch",
    line_by="timing_environment",
    filters={
        "preset": "ang", "implementation": "qml_torch",
        "gradient_method": "SPSA", "n_qubits": 4, "randomness": 0,
    },
    ax=axes[0, 1],
)
axes[0, 1].set_yscale("log")
axes[0, 1].set_title("Timing by execution type")

plot_metric_by_category_field(
    primary_timing,
    x_field="gradient_method",
    metric_name="median_time_per_epoch",
    line_by="timing_environment",
    filters={
        "preset": "ang", "implementation": "qml_torch",
        "execution_type": "noiseless", "n_qubits": 4, "randomness": 0,
    },
    ax=axes[1, 0],
)
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("Timing by gradient method")

plot_metric_by_category_field(
    primary_timing,
    x_field="randomness",
    metric_name="median_time_per_epoch",
    line_by="timing_environment",
    filters={
        "preset": "ang", "implementation": "qml_torch",
        "execution_type": "noiseless", "gradient_method": "SPSA",
        "n_qubits": 4, "randomness": (0, 1),
    },
    ax=axes[1, 1],
)
axes[1, 1].set_yscale("log")
axes[1, 1].set_title("Timing by rand0/rand1")
finish_figure(fig, "01a_timing_main_factors")


#### 1.2 Experimental limitations and feasibility

This subsection records the known resource boundary for each omitted or unfinished experiment class. It deliberately excludes `fake_real`, which is an implementation-validation mode rather than a feasibility result. References come only from the current convergence batteries; files ending in `_old` and all dedicated timing batteries are excluded.

The categories distinguish time cost, unavailable real-hardware execution time, statevector transpilation limits, and CPU memory limits. They should be interpreted as experimental design constraints, not model-quality outcomes.


In [ ]:
from qgan_v2.config.battery import build_config_combinations, load_battery_file
from qgan_v2.storage.paths import get_config_filename


convergence_battery_files = sorted(
    path for path in (qgan_dir / "configs" / "batteries" / "train").glob("*.yaml")
    if not path.stem.endswith("_old")
    and not path.stem.startswith("train_times")
)
assert all("_old" not in path.stem for path in convergence_battery_files)
print("non-old convergence battery references:")
for path in convergence_battery_files:
    print(" -", path.name)

limitation_cases = [
    {
        "n_qubits": 4,
        "experiment": "noisy PSR",
        "limitation": "time-expensive",
        "detail": "Parameter-shift noisy convergence did not finish.",
        "battery_reference": "train_psr_noisy_q4_*",
    },
    {
        "n_qubits": 4,
        "experiment": "real hardware",
        "limitation": "execution time unavailable",
        "detail": "Only the two short RH case-study runs had hardware resources.",
        "battery_reference": "train_conv_rh.yaml",
    },
    {
        "n_qubits": 8,
        "experiment": "noisy PSR",
        "limitation": "time-expensive",
        "detail": "Parameter-shift noisy convergence did not finish.",
        "battery_reference": "train_psr_noisy_q8_*",
    },
    {
        "n_qubits": 16,
        "experiment": "noisy simulation on GPU",
        "limitation": "time-expensive",
        "detail": "Projected execution exceeded the available GPU allocation.",
        "battery_reference": "train_conv_gpu.yaml",
    },
    {
        "n_qubits": 16,
        "experiment": "amplitude preset",
        "limitation": "not transpilable",
        "detail": "The statevector preparation circuit was too large/deep to transpile.",
        "battery_reference": "train_conv_cpu.yaml / train_conv_gpu.yaml",
    },
    {
        "n_qubits": 16,
        "experiment": "noisy simulation on CPU",
        "limitation": "out of memory",
        "detail": "The density-matrix simulation exceeded CPU memory.",
        "battery_reference": "train_conv_cpu.yaml",
    },
]
limitation_cases.sort(key=lambda row: (row["n_qubits"], row["experiment"]))
show_rows(limitation_cases)

reference_run_paths = set()
for battery_file in convergence_battery_files:
    default_config, variable_groups = load_battery_file(battery_file)
    for variable_values in variable_groups.values():
        for config in build_config_combinations(default_config, variable_values):
            if config["experiment"]["execution_type"] != "fake_real":
                reference_run_paths.add(get_config_filename(config).parent.resolve())

feasibility_results = [
    run for run in raw_convergence
    if run.path.resolve() in reference_run_paths
    and run.metadata.get("execution_type") != "fake_real"
]

qubit_order = (4, 8, 16)
limitation_order = (
    "time-expensive",
    "execution time unavailable",
    "not transpilable",
    "out of memory",
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_feasibility_matrix(
    feasibility_results,
    row_field="n_qubits",
    column_field="execution_type",
    filters={"implementation": "qml_torch"},
    ax=axes[0],
)
axes[0].set_yticks(
    np.arange(len(qubit_order)),
    [f"q{value}" for value in qubit_order],
)
axes[0].set_title("Completed/requested non-old battery runs")

bottom = np.zeros(len(qubit_order))
for limitation in limitation_order:
    counts = np.asarray([
        sum(
            row["n_qubits"] == n_qubits and row["limitation"] == limitation
            for row in limitation_cases
        )
        for n_qubits in qubit_order
    ])
    axes[1].bar(
        np.arange(len(qubit_order)), counts, bottom=bottom,
        label=limitation,
    )
    bottom += counts
axes[1].set_xticks(np.arange(len(qubit_order)), [f"q{value}" for value in qubit_order])
axes[1].set_ylabel("Known limited experiment classes")
axes[1].set_title("Feasibility limitation counts")
axes[1].legend(fontsize=8)
axes[1].grid(True, axis="y", alpha=0.25)
finish_figure(fig, "01b_experimental_limitations_and_feasibility")


### 2. Presets

This section establishes how the three data/encoding presets learn and then links the quantitative curves to generated outputs.


#### 2.1 Learning dynamics and best results

The comparison fixes noiseless SPSA, `q4`, `rand0`, and seeds 0–2. Each preset has its own column because `base` minimizes KL divergence over a target probability distribution, while `ang` and `amp` use an image-gradient score.

In each column, generator and discriminator losses are joined in the upper panel and evaluation appears below. Summary panels show only best evaluation and the epoch at which it occurs; point color identifies the seed.


In [ ]:
PRESET_FOCUS = {
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "gradient_method": "PSR",
    "n_qubits": 4,
    "randomness": 0,
}
preset_runs = filter_results(main_convergence, **PRESET_FOCUS)
fig, axes = plt.subplots(2, 3, figsize=(16, 8.5), sharex="col")
for column, preset in enumerate(("base", "ang", "amp")):
    selected = filter_results(preset_runs, preset=preset)
    plot_training_dynamics_comparison(
        selected,
        compare_by="preset",
        center="median",
        spread="iqr",
        axes=axes[:, column],
    )
    axes[0, column].set_title(f"{preset} preset")
fig.suptitle("Preset learning dynamics: individual seeds, median, and IQR")
finish_figure(fig, "02a_preset_learning_dynamics")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for column, preset in enumerate(("base", "ang", "amp")):
    selected = filter_results(preset_runs, preset=preset)
    plot_final_metric(
        selected, compare_by="preset", metric_name="best_eval",
        point_color_by="seed", ax=axes[0, column],
    )
    axes[0, column].set_title(f"{preset}: best evaluation")
    plot_final_metric(
        selected, compare_by="preset", metric_name="epoch_of_best_eval",
        point_color_by="seed", ax=axes[1, column],
    )
    axes[1, column].set_title(f"{preset}: epoch of best")
fig.suptitle("Best preset results across seeds")
finish_figure(fig, "02b_preset_best_results")

show_rows(
    grouped_performance_table(
        preset_runs,
        fields=("eval_method", "preset", "n_qubits", "gradient_method", "randomness"),
    ),
    columns=(
        "eval_method", "preset", "completed_runs", "seeds",
        "median_best_eval", "q25_best_eval", "q75_best_eval",
        "median_epoch_of_best_eval",
    ),
)


#### 2.2 Initial, best, and last generated outputs

One completed noiseless `q4` run is chosen automatically for each preset using its best evaluation score. If runs tie, gradient methods are preferred in the requested order: PSR, REG, then SPSA. The initial, best-checkpoint, and final-checkpoint outputs are rendered from that same selected run, making the full qualitative change and any regression after the optimum directly visible.


In [ ]:
output_candidates = filter_results(
    main_convergence,
    implementation="qml_torch",
    execution_type="noiseless",
    n_qubits=4,
)
gradient_tie_priority = {"PSR": 0, "REG": 1, "SPSA": 2}
preset_output_runs = {}
for preset in ("base", "ang", "amp"):
    candidates = filter_results(output_candidates, preset=preset)
    ranked = []
    for run in candidates:
        summary = results_table([run])[0]
        if np.isfinite(summary["best_eval"]):
            ranked.append((
                float(summary["best_eval"]),
                gradient_tie_priority.get(run.metadata.get("gradient_method"), 99),
                int(run.metadata.get("seed") or 0),
                run.run_id,
                run,
                summary,
            ))
    if ranked:
        preset_output_runs[preset] = min(ranked)[-2:]

preset_output_manifest = []
for preset, (run, summary) in preset_output_runs.items():
    preset_output_manifest.append({
        "preset": preset,
        "run_id": run.run_id,
        "gradient_method": summary["gradient_method"],
        "randomness": summary["randomness"],
        "seed": summary["seed"],
        "evaluation_metric": summary["eval_method"],
        "best_eval": summary["best_eval"],
        "epoch_of_best_eval": summary["epoch_of_best_eval"],
        "last_eval": summary["final_eval"],
        "config_file": str(run.path / "config.yaml"),
    })
show_rows(preset_output_manifest)


In [ ]:
RENDER_PRESET_OUTPUTS = True
QUALITATIVE_RANDOM_SEED = 0

if RENDER_PRESET_OUTPUTS:
    rendered_output_manifest = []
    for preset, (run, _) in preset_output_runs.items():
        for parameter_set in ("initial", "best", "last"):
            fig, record = plot_generated_output(
                run.path / "config.yaml",
                parameter_set=parameter_set,
                random_seed=QUALITATIVE_RANDOM_SEED,
            )
            rendered_output_manifest.append({"preset": preset, **record})
            finish_figure(fig, f"02c_{preset}_{parameter_set}_generated_output")
    show_rows(rendered_output_manifest)
else:
    print("Set RENDER_PRESET_OUTPUTS = True to create initial, best, and last output figures.")


### 3. Execution-Type Comparison

Simulation provides the replicated comparison; the much smaller real-hardware sample is separated so its evidential limits remain explicit.


#### 3.1 Noiseless versus noisy simulation

This comparison uses matched `q4`, SPSA, `rand0`, seeds 0–2. Each preset remains in its own metric column. Learning curves use the joined-loss/evaluation layout; summaries retain only best evaluation and best epoch.


In [ ]:
SIMULATION_EXECUTION_FOCUS = {
    "implementation": "qml_torch",
    "gradient_method": "SPSA",
    "n_qubits": 4,
    "randomness": 0,
    "execution_type": ("noiseless", "noisy"),
}
simulation_execution_runs = filter_results(main_convergence, **SIMULATION_EXECUTION_FOCUS)
fig, axes = plt.subplots(2, 3, figsize=(16, 8.5), sharex="col")
for column, preset in enumerate(("base", "ang", "amp")):
    selected = filter_results(simulation_execution_runs, preset=preset)
    plot_training_dynamics_comparison(
        selected, compare_by="execution_type", axes=axes[:, column]
    )
    axes[0, column].set_title(preset)
fig.suptitle("Noiseless versus noisy learning dynamics")
finish_figure(fig, "03a_noiseless_noisy_dynamics")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for column, preset in enumerate(("base", "ang", "amp")):
    selected = filter_results(simulation_execution_runs, preset=preset)
    plot_final_metric(
        selected, compare_by="execution_type", metric_name="best_eval",
        point_color_by="seed", ax=axes[0, column],
    )
    axes[0, column].set_title(f"{preset}: best evaluation")
    plot_final_metric(
        selected, compare_by="execution_type", metric_name="epoch_of_best_eval",
        point_color_by="seed", ax=axes[1, column],
    )
    axes[1, column].set_title(f"{preset}: epoch of best")
fig.suptitle("Noiseless versus noisy best results")
finish_figure(fig, "03b_noiseless_noisy_best_results")


#### 3.2 Noiseless, noisy, and real hardware

The two available real-QPU runs are compared against matching simulator runs truncated to the same observed epoch budget. This is behavioral evidence for those runs, not a replicated estimate of hardware performance.


In [ ]:
real_runs = filter_results(
    hardware_results,
    execution_type="real",
    implementation="qml_torch",
    preset="base",
    n_qubits=4,
    randomness=0,
)
hardware_cases = {}
for gradient_method in unique_values(real_runs, "gradient_method"):
    hardware_group = filter_results(real_runs, gradient_method=gradient_method)
    matched_simulators = filter_results(
        main_convergence,
        preset="base",
        implementation="qml_torch",
        gradient_method=gradient_method,
        n_qubits=4,
        randomness=0,
        eval_method="kl",
    )
    budget = max(len(run.eval) for run in hardware_group)
    hardware_cases[gradient_method] = truncate_results(
        [*matched_simulators, *hardware_group],
        max_epoch=budget - 1,
    )

if hardware_cases:
    fig, axes = plt.subplots(
        2, len(hardware_cases),
        figsize=(7 * len(hardware_cases), 7.5), sharex="col",
    )
    axes = np.asarray(axes).reshape(2, -1)
    for column, (gradient_method, selected) in enumerate(hardware_cases.items()):
        plot_training_dynamics_comparison(
            selected, compare_by="execution_type", axes=axes[:, column]
        )
        axes[0, column].set_title(f"{gradient_method} hardware case")
    fig.suptitle("Matched simulator and real-QPU learning dynamics")
    finish_figure(fig, "03c_real_hardware_dynamics")

    fig, axes = plt.subplots(2, len(hardware_cases), figsize=(6 * len(hardware_cases), 8))
    axes = np.asarray(axes).reshape(2, -1)
    for column, (gradient_method, selected) in enumerate(hardware_cases.items()):
        plot_final_metric(
            selected, compare_by="execution_type", metric_name="best_eval",
            point_color_by="seed", ax=axes[0, column],
        )
        axes[0, column].set_title(f"{gradient_method}: best evaluation")
        plot_final_metric(
            selected, compare_by="execution_type", metric_name="epoch_of_best_eval",
            point_color_by="seed", ax=axes[1, column],
        )
        axes[1, column].set_title(f"{gradient_method}: epoch of best")
    finish_figure(fig, "03d_real_hardware_best_results")

show_rows(results_table(real_runs), columns=(
    "run_id", "gradient_method", "completed_epochs", "best_eval",
    "epoch_of_best_eval", "median_time_per_epoch",
))


### 4. Gradient Method Comparison

SPSA, PSR, and REG are compared with preset, execution type, qubits, randomness, and evaluation metric fixed. Timing is absent here because it is analyzed in Section 1.


In [ ]:
GRADIENT_FOCUS = {
    "preset": "ang",
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "n_qubits": 4,
    "randomness": 0,
    "eval_method": "gradient",
}
gradient_runs = filter_results(main_convergence, **GRADIENT_FOCUS)
dynamics_figure(
    gradient_runs, "gradient_method",
    "Learning dynamics by gradient method", "04a_gradient_dynamics",
)
best_results_figure(
    gradient_runs, "gradient_method",
    "Best results by gradient method", "04b_gradient_best_results",
)


### 5. Randomness Effect

Randomness is first studied as a complete five-level sweep, then with a wider matched `rand0`/`rand1` comparison.


#### 5.1 Complete five-level randomness sweep

The principal analysis requires all five values (`0`, `0.1`, `0.25`, `0.5`, `1`) for seeds 0–2. Evaluation step volatility is the median absolute change between consecutive evaluation scores; larger values indicate a more erratic learning trajectory.


In [ ]:
RANDOMNESS_LEVELS = (0, 0.1, 0.25, 0.5, 1)
RANDOMNESS_SWEEP_INDEX = 0
randomness_sweeps = factor_sweep_groups(
    main_convergence,
    factor="randomness",
    required_levels=RANDOMNESS_LEVELS,
    required_seeds=(0, 1, 2),
)
ordered_sweeps = sorted(randomness_sweeps.items(), key=lambda item: str(item[0]))
print("complete five-level × three-seed sweeps:", len(ordered_sweeps))

if ordered_sweeps:
    _, randomness_main = ordered_sweeps[RANDOMNESS_SWEEP_INDEX]
    exemplar = randomness_main[0]
    print("selected sweep:", {
        field: exemplar.metadata.get(field)
        for field in ("preset", "gradient_method", "n_qubits", "execution_type", "eval_method")
    })
    dynamics_figure(
        randomness_main, "randomness",
        "Learning dynamics across randomness strength",
        "05a_randomness_complete_dynamics",
    )

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, metric_name, title in (
        (axes[0], "best_eval", "Best evaluation"),
        (axes[1], "epoch_of_best_eval", "Epoch of best evaluation"),
        (axes[2], "evaluation_step_volatility", "Evaluation step volatility"),
    ):
        plot_final_metric(
            randomness_main, compare_by="randomness", metric_name=metric_name,
            point_color_by="seed", ax=ax,
        )
        ax.set_title(title)
    finish_figure(fig, "05b_randomness_complete_best_and_volatility")
else:
    print("No fully matched randomness sweep is available.")


#### 5.2 Wider paired rand0/rand1 evidence

The wider dataset is reduced to matched `rand0` and `rand1` pairs. Panels show the change in best evaluation, best epoch, and evaluation step volatility. Zero means no effect; positive score or volatility deltas mean worse or bumpier behavior.


In [ ]:
randomness_pair_fields = tuple(
    field for field in SCIENTIFIC_PAIR_FIELDS if field != "randomness"
)
extra_randomness_filters = {
    "preset": "ang",
    "implementation": "qml_torch",
    "execution_type": "noiseless",
    "eval_method": "gradient",
}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric_name, title in (
    (axes[0], "best_eval", "Δ best evaluation"),
    (axes[1], "epoch_of_best_eval", "Δ epoch of best"),
    (axes[2], "evaluation_step_volatility", "Δ evaluation volatility"),
):
    plot_paired_delta(
        main_convergence,
        baseline_filters={**extra_randomness_filters, "randomness": 0},
        treatment_filters={**extra_randomness_filters, "randomness": 1},
        metric_name=metric_name,
        pair_fields=randomness_pair_fields,
        x_field="gradient_method",
        value="delta",
        ax=ax,
    )
    ax.set_title(title)
fig.suptitle("Paired rand1 − rand0 effects")
finish_figure(fig, "05c_randomness_rand0_rand1_paired")


### 6. Scaling Analysis

Scaling is split into preset, execution type, gradient method, and randomness. Each subsection contains representative joined-loss/evaluation dynamics plus best-evaluation and best-epoch scaling across qubit counts. Parameter-count and circuit-width figures are intentionally reserved for the experimental-setup chapter.


#### 6.1 Scaling by preset

Noiseless SPSA and `rand0` are fixed. Each preset is faceted to respect its evaluation metric; qubit count is the compared curve within each facet.


In [ ]:
preset_scaling_runs = filter_results(
    main_convergence,
    implementation="qml_torch",
    execution_type="noiseless",
    gradient_method="SPSA",
    randomness=0,
)
fig, axes = plt.subplots(2, 3, figsize=(16, 8.5), sharex="col")
for column, preset in enumerate(("base", "ang", "amp")):
    allowed_qubits = (4, 8) if preset == "amp" else (4, 8, 16)
    selected = filter_results(
        preset_scaling_runs, preset=preset, n_qubits=allowed_qubits
    )
    plot_training_dynamics_comparison(
        selected, compare_by="n_qubits", axes=axes[:, column]
    )
    axes[0, column].set_title(preset)
fig.suptitle("Learning dynamics as qubit count scales, by preset")
finish_figure(fig, "06a_preset_scaling_dynamics")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for column, preset in enumerate(("base", "ang", "amp")):
    selected = filter_results(preset_scaling_runs, preset=preset)
    plot_metric_by_numeric_field(
        selected, x_field="n_qubits", metric_name="best_eval", ax=axes[0, column]
    )
    axes[0, column].set_title(f"{preset}: best evaluation")
    plot_metric_by_numeric_field(
        selected, x_field="n_qubits", metric_name="epoch_of_best_eval", ax=axes[1, column]
    )
    axes[1, column].set_title(f"{preset}: epoch of best")
finish_figure(fig, "06b_preset_scaling_best_results")


#### 6.2 Scaling by execution type

Representative dynamics use `ang`, SPSA, `rand0`, `q4`; the summary shows how noiseless and noisy behavior changes from `q4` to `q8`. No noisy `q16` result is inferred.


In [ ]:
execution_scaling_runs = filter_results(
    main_convergence,
    preset="ang", implementation="qml_torch", gradient_method="SPSA",
    randomness=0, execution_type=("noiseless", "noisy"),
)
dynamics_figure(
    filter_results(execution_scaling_runs, n_qubits=4),
    "execution_type", "Representative execution-type dynamics at q4",
    "06c_execution_scaling_dynamics",
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_metric_by_numeric_field(
    execution_scaling_runs, x_field="n_qubits", metric_name="best_eval",
    line_by="execution_type", ax=axes[0],
)
axes[0].set_title("Best evaluation scaling")
plot_metric_by_numeric_field(
    execution_scaling_runs, x_field="n_qubits", metric_name="epoch_of_best_eval",
    line_by="execution_type", ax=axes[1],
)
axes[1].set_title("Best-epoch scaling")
finish_figure(fig, "06d_execution_scaling_best_results")


#### 6.3 Scaling by gradient method

Representative dynamics use `ang`, noiseless, `rand0`, `q4`; the summary compares SPSA, PSR, and REG as qubit count increases where completed runs exist.


In [ ]:
gradient_scaling_runs = filter_results(
    main_convergence,
    preset="ang", implementation="qml_torch", execution_type="noiseless",
    randomness=0,
)
dynamics_figure(
    filter_results(gradient_scaling_runs, n_qubits=4),
    "gradient_method", "Representative gradient dynamics at q4",
    "06e_gradient_scaling_dynamics",
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_metric_by_numeric_field(
    gradient_scaling_runs, x_field="n_qubits", metric_name="best_eval",
    line_by="gradient_method", ax=axes[0],
)
axes[0].set_title("Best evaluation scaling")
plot_metric_by_numeric_field(
    gradient_scaling_runs, x_field="n_qubits", metric_name="epoch_of_best_eval",
    line_by="gradient_method", ax=axes[1],
)
axes[1].set_title("Best-epoch scaling")
finish_figure(fig, "06f_gradient_scaling_best_results")


#### 6.4 Scaling by randomness

Representative dynamics compare `rand0` and `rand1` for `ang`, noiseless SPSA, `q4`; the summary follows both levels over increasing qubit count.


In [ ]:
randomness_scaling_runs = filter_results(
    main_convergence,
    preset="ang", implementation="qml_torch", execution_type="noiseless",
    gradient_method="SPSA", randomness=(0, 1),
)
dynamics_figure(
    filter_results(randomness_scaling_runs, n_qubits=4),
    "randomness", "Representative rand0/rand1 dynamics at q4",
    "06g_randomness_scaling_dynamics",
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_metric_by_numeric_field(
    randomness_scaling_runs, x_field="n_qubits", metric_name="best_eval",
    line_by="randomness", ax=axes[0],
)
axes[0].set_title("Best evaluation scaling")
plot_metric_by_numeric_field(
    randomness_scaling_runs, x_field="n_qubits", metric_name="epoch_of_best_eval",
    line_by="randomness", ax=axes[1],
)
axes[1].set_title("Best-epoch scaling")
finish_figure(fig, "06h_randomness_scaling_best_results")


### 7. Implementation and Packing Validation

This final section only demonstrates that each implementation path trains. Fake-real `q4`, SPSA, `rand0`, seed-0 traces cover `qml_torch`, `runtime_packed/separate`, and valid `runtime_packed/joined` cases. No deep quality, timing, or statistical claim is made from one seed.


In [ ]:
fake_validation = filter_results(
    validation_results,
    execution_type="fake_real",
    n_qubits=4,
    gradient_method="SPSA",
    randomness=0,
    seed=0,
)
fig, axes = plt.subplots(2, 3, figsize=(16, 8.5), sharex="col")
for column, preset in enumerate(("base", "ang", "amp")):
    selected = filter_results(fake_validation, preset=preset)
    plot_training_dynamics_comparison(
        selected, compare_by="implementation_packing", axes=axes[:, column]
    )
    axes[0, column].set_title(preset)
fig.suptitle("Implementation and packing validation traces")
finish_figure(fig, "07_implementation_packing_validation")

show_rows(results_table(fake_validation), columns=(
    "run_id", "preset", "implementation_packing", "completed_epochs",
    "best_eval", "epoch_of_best_eval",
))
